In [1]:
# ----------------------------
# Cell 2 — Input API Key
# ----------------------------
import os

GROQ_API_KEY = input("Enter your Groq API key: ").strip()
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("API key set successfully!")

API key set successfully!


In [3]:
# ----------------------------
# Cell 3 — Imports
# ----------------------------
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, RemoveMessage
from typing import Literal, Dict, Any, Optional
import sqlite3
import json
import uuid
from datetime import datetime

In [4]:
# ----------------------------
# Cell 4 — Initialize Groq Model
# ----------------------------
chat = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    max_tokens=200,
    timeout=60,
)

In [5]:
# ----------------------------
# Cell 5 — Define State with Memory Fields
# ----------------------------
class State(MessagesState):
    user_id: str
    session_id: str
    summary: str
    last_updated: str
    metadata: Dict[str, Any]

In [6]:
# ----------------------------
# Cell 6 — Setup Memory Saver (Short-term Memory)
# ----------------------------
# In-memory checkpoint saver for short-term memory
memory_saver = MemorySaver()

print("Short-term memory saver initialized")

Short-term memory saver initialized


In [7]:
# ----------------------------
# Cell 7 — Setup SQLite Long-term Memory
# ----------------------------
# Create SQLite database for long-term memory
conn = sqlite3.connect("conversation_memory.db", check_same_thread=False)
cursor = conn.cursor()

# Create tables for storing conversations and summaries
cursor.execute("""
CREATE TABLE IF NOT EXISTS conversations (
    id TEXT PRIMARY KEY,
    user_id TEXT,
    session_id TEXT,
    messages TEXT,
    summary TEXT,
    created_at TEXT,
    updated_at TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS user_memory (
    user_id TEXT PRIMARY KEY,
    preferences TEXT,
    facts TEXT,
    last_summary TEXT,
    updated_at TEXT
)
""")

conn.commit()
print("SQLite long-term memory database initialized")

SQLite long-term memory database initialized


In [8]:
# ----------------------------
# Cell 8 — Helper Functions for Long-term Memory
# ----------------------------
def save_conversation_to_db(user_id: str, session_id: str, messages: list, summary: str = ""):
    """Save conversation to SQLite database"""
    conv_id = str(uuid.uuid4())
    messages_json = json.dumps([{"type": m.type, "content": m.content} for m in messages])
    now = datetime.now().isoformat()
    
    cursor.execute("""
    INSERT OR REPLACE INTO conversations 
    (id, user_id, session_id, messages, summary, created_at, updated_at)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (conv_id, user_id, session_id, messages_json, summary, now, now))
    conn.commit()
    return conv_id

def load_conversation_from_db(user_id: str, session_id: str, limit: int = 5):
    """Load recent conversations from database"""
    cursor.execute("""
    SELECT messages, summary FROM conversations 
    WHERE user_id = ? AND session_id = ?
    ORDER BY updated_at DESC LIMIT ?
    """, (user_id, session_id, limit))
    
    results = cursor.fetchall()
    conversations = []
    for messages_json, summary in results:
        messages_data = json.loads(messages_json)
        messages = []
        for msg in messages_data:
            if msg["type"] == "human":
                messages.append(HumanMessage(content=msg["content"]))
            elif msg["type"] == "ai":
                messages.append(AIMessage(content=msg["content"]))
        conversations.append({"messages": messages, "summary": summary})
    
    return conversations

def save_user_memory(user_id: str, preferences: dict = None, facts: dict = None, summary: str = ""):
    """Save user-specific memory"""
    now = datetime.now().isoformat()
    
    # Get existing memory
    cursor.execute("SELECT preferences, facts FROM user_memory WHERE user_id = ?", (user_id,))
    existing = cursor.fetchone()
    
    if existing:
        current_prefs = json.loads(existing[0]) if existing[0] else {}
        current_facts = json.loads(existing[1]) if existing[1] else {}
    else:
        current_prefs = {}
        current_facts = {}
    
    # Update with new data
    if preferences:
        current_prefs.update(preferences)
    if facts:
        current_facts.update(facts)
    
    prefs_json = json.dumps(current_prefs)
    facts_json = json.dumps(current_facts)
    
    cursor.execute("""
    INSERT OR REPLACE INTO user_memory 
    (user_id, preferences, facts, last_summary, updated_at)
    VALUES (?, ?, ?, ?, ?)
    """, (user_id, prefs_json, facts_json, summary, now))
    conn.commit()

def load_user_memory(user_id: str):
    """Load user-specific memory"""
    cursor.execute("SELECT preferences, facts, last_summary FROM user_memory WHERE user_id = ?", (user_id,))
    result = cursor.fetchone()
    
    if result:
        preferences = json.loads(result[0]) if result[0] else {}
        facts = json.loads(result[1]) if result[1] else {}
        summary = result[2] if result[2] else ""
        return {"preferences": preferences, "facts": facts, "summary": summary}
    return {"preferences": {}, "facts": {}, "summary": ""}

In [9]:
# ----------------------------
# Cell 9 — Node Functions
# ----------------------------
def start_session(state: State) -> State:
    """Initialize a new session with user context"""
    print(f"\n-------> ENTERING start_session")
    print(f"User ID: {state['user_id']}, Session ID: {state['session_id']}")
    
    # Load user memory from long-term storage
    user_memory = load_user_memory(state['user_id'])
    
    # Load recent conversations
    recent_convs = load_conversation_from_db(state['user_id'], state['session_id'])
    
    # Create context message with memory
    context = f"Welcome back! I remember you."
    if user_memory['preferences']:
        context += f" Your preferences: {user_memory['preferences']}"
    if user_memory['facts']:
        context += f" Facts about you: {user_memory['facts']}"
    if user_memory['summary']:
        context += f" Previous conversation summary: {user_memory['summary']}"
    
    welcome_msg = AIMessage(content=context)
    
    return State(
        messages=[welcome_msg],
        user_id=state['user_id'],
        session_id=state['session_id'],
        summary=user_memory['summary'],
        last_updated=datetime.now().isoformat(),
        metadata={"user_memory": user_memory, "recent_conversations": len(recent_convs)}
    )

def get_user_input(state: State) -> State:
    """Get input from user"""
    print(f"\n-------> ENTERING get_user_input")
    print("What would you like to ask?")
    user_input = input("> ").strip()
    
    return State(
        messages=[HumanMessage(content=user_input)],
        user_id=state['user_id'],
        session_id=state['session_id'],
        summary=state.get('summary', ''),
        last_updated=datetime.now().isoformat(),
        metadata=state.get('metadata', {})
    )

def process_with_memory(state: State) -> State:
    """Process user input with memory context"""
    print(f"\n-------> ENTERING process_with_memory")
    
    # Build context with memory
    system_content = "You are a helpful assistant with memory."
    if state.get('summary'):
        system_content += f"\nConversation summary: {state['summary']}"
    
    if 'user_memory' in state.get('metadata', {}):
        mem = state['metadata']['user_memory']
        if mem.get('preferences'):
            system_content += f"\nUser preferences: {mem['preferences']}"
        if mem.get('facts'):
            system_content += f"\nKnown facts: {mem['facts']}"
    
    system_msg = SystemMessage(content=system_content)
    
    # Get response
    response = chat.invoke([system_msg] + state['messages'])
    print(f"Assistant: {response.content}")
    
    return State(
        messages=[response],
        user_id=state['user_id'],
        session_id=state['session_id'],
        summary=state.get('summary', ''),
        last_updated=datetime.now().isoformat(),
        metadata=state.get('metadata', {})
    )

def check_continue(state: State) -> State:
    """Ask if user wants to continue"""
    print(f"\n-------> ENTERING check_continue")
    print("Would you like to continue? (yes/no)")
    user_input = input("> ").strip().lower()
    
    return State(
        messages=[HumanMessage(content=user_input)],
        user_id=state['user_id'],
        session_id=state['session_id'],
        summary=state.get('summary', ''),
        last_updated=datetime.now().isoformat(),
        metadata=state.get('metadata', {})
    )

def summarize_and_store(state: State) -> State:
    """Summarize conversation and store in long-term memory"""
    print(f"\n-------> ENTERING summarize_and_store")
    
    # Create summary of conversation
    conversation_text = ""
    for msg in state['messages'][-6:]:  # Last 6 messages
        conversation_text += f"{msg.type}: {msg.content}\n"
    
    summary_prompt = f"""
    Summarize this conversation concisely:
    {conversation_text}
    
    Previous summary: {state.get('summary', '')}
    
    Provide updated summary:
    """
    
    summary_response = chat.invoke([HumanMessage(content=summary_prompt)])
    new_summary = summary_response.content
    
    # Extract preferences or facts from conversation
    extract_prompt = f"""
    From this conversation, extract:
    1. User preferences (likes, dislikes, preferences)
    2. Important facts about the user
    
    Conversation: {conversation_text}
    
    Return as JSON with keys 'preferences' and 'facts':
    """
    
    extract_response = chat.invoke([HumanMessage(content=extract_prompt)])
    
    try:
        extracted = json.loads(extract_response.content)
    except:
        extracted = {"preferences": {}, "facts": {}}
    
    # Save to long-term memory
    save_conversation_to_db(state['user_id'], state['session_id'], state['messages'], new_summary)
    save_user_memory(state['user_id'], 
                    preferences=extracted.get('preferences'),
                    facts=extracted.get('facts'),
                    summary=new_summary)
    
    print(f"Summary updated and stored in database")
    
    # Keep only recent messages to manage context window
    remove_msgs = [RemoveMessage(id=msg.id) for msg in state['messages'][:-4] if hasattr(msg, 'id') and msg.id]
    
    return State(
        messages=remove_msgs + state['messages'][-4:],
        user_id=state['user_id'],
        session_id=state['session_id'],
        summary=new_summary,
        last_updated=datetime.now().isoformat(),
        metadata=state.get('metadata', {})
    )

In [10]:
# ----------------------------
# Cell 10 — Routing Functions
# ----------------------------
def route_after_input(state: State) -> Literal["process_with_memory", "__end__"]:
    """Route based on user input"""
    if state['messages'] and state['messages'][-1].content.lower() == 'quit':
        return "__end__"
    return "process_with_memory"

def route_after_response(state: State) -> Literal["check_continue", "summarize_and_store"]:
    """Route after processing response"""
    # Check if we need to summarize (every 3 turns)
    if len(state['messages']) > 6:
        return "summarize_and_store"
    return "check_continue"

def route_continue(state: State) -> Literal["get_user_input", "summarize_and_store", "__end__"]:
    """Route based on continue choice"""
    last_msg = state['messages'][-1].content.lower() if state['messages'] else ""
    
    if last_msg == 'yes':
        return "get_user_input"
    elif last_msg == 'no':
        return "summarize_and_store"
    else:
        return "__end__"

In [11]:
# ----------------------------
# Cell 11 — Build the Graph
# ----------------------------
# Create graph with MemorySaver for checkpoints
graph = StateGraph(State)

# Add nodes
graph.add_node("start_session", start_session)
graph.add_node("get_user_input", get_user_input)
graph.add_node("process_with_memory", process_with_memory)
graph.add_node("check_continue", check_continue)
graph.add_node("summarize_and_store", summarize_and_store)

# Add edges
graph.add_edge(START, "start_session")
graph.add_edge("start_session", "get_user_input")

# Conditional edges
graph.add_conditional_edges("get_user_input", route_after_input)
graph.add_conditional_edges("process_with_memory", route_after_response)
graph.add_conditional_edges("check_continue", route_continue)

# Fixed edges
graph.add_edge("summarize_and_store", "get_user_input")

# Compile with MemorySaver for checkpoints
graph_compiled = graph.compile(checkpointer=memory_saver)

print("Graph compiled with MemorySaver checkpointer")

Graph compiled with MemorySaver checkpointer


In [12]:
# ----------------------------
# Cell 12 — Display Graph Structure
# ----------------------------
print("Graph Structure:")
print(graph_compiled.get_graph().draw_ascii())

Graph Structure:
                                        +-----------+                                        
                                        | __start__ |                                        
                                        +-----------+                                        
                                              *                                              
                                              *                                              
                                              *                                              
                                      +---------------+                                      
                                      | start_session |                                      
                                      +---------------+                                      
                                              *                                              
                                           

In [13]:
# ----------------------------
# Cell 13 — Run the Graph with Memory
# ----------------------------
import uuid
from langgraph.checkpoint.base import BaseCheckpointSaver

# Create unique user and session IDs
user_id = input("Enter your user ID (or press Enter for new user): ").strip()
if not user_id:
    user_id = f"user_{uuid.uuid4().hex[:8]}"
    print(f"Created new user ID: {user_id}")

session_id = f"session_{uuid.uuid4().hex[:8]}"
thread_id = f"thread_{uuid.uuid4().hex[:8]}"

print(f"Session ID: {session_id}")
print(f"Thread ID: {thread_id}")

# Initial state
initial_state = State(
    messages=[],
    user_id=user_id,
    session_id=session_id,
    summary="",
    last_updated=datetime.now().isoformat(),
    metadata={}
)

# Configuration for checkpointer
config = {"configurable": {"thread_id": thread_id}}

# Run the graph
print("\n" + "="*50)
print("Starting Memory-Enabled Conversation")
print("="*50)
print("Type 'quit' to exit at any prompt\n")

try:
    final_state = graph_compiled.invoke(initial_state, config)
    print("\n" + "="*50)
    print("Conversation completed")
    print("="*50)
except Exception as e:
    print(f"Error: {e}")

Created new user ID: user_4ef024dc
Session ID: session_5b9a997e
Thread ID: thread_8f83eb87

Starting Memory-Enabled Conversation
Type 'quit' to exit at any prompt


-------> ENTERING start_session
User ID: user_4ef024dc, Session ID: session_5b9a997e

-------> ENTERING get_user_input
What would you like to ask?

-------> ENTERING process_with_memory
Assistant: Not much! Just here and ready to help with anything you need. What's on your mind? Want to chat about something or need assistance with a particular topic?

-------> ENTERING check_continue
Would you like to continue? (yes/no)

-------> ENTERING get_user_input
What would you like to ask?

-------> ENTERING process_with_memory
Assistant: You're referring to Mark Zuckerberg, the co-founder and CEO of Facebook. I'm assuming you're talking about the situation with Eduardo Saverin, one of Facebook's co-founders.

To give you some background, Eduardo Saverin was a close friend of Mark Zuckerberg's and was also a co-founder of Facebook. 

In [14]:
# ----------------------------
# Cell 14 — StateSnapshot Demonstration
# ----------------------------
from langgraph.checkpoint.base import BaseCheckpointSaver

print("\n" + "="*50)
print("StateSnapshot Demonstration")
print("="*50)

# Get all checkpoints for this thread
try:
    # Get latest state snapshot
    latest_state = graph_compiled.get_state(config)
    print(f"\nLatest State:")
    print(f"  - Next node: {latest_state.next}")
    print(f"  - Messages count: {len(latest_state.values.get('messages', []))}")
    print(f"  - Summary: {latest_state.values.get('summary', '')[:100]}...")
    print(f"  - User ID: {latest_state.values.get('user_id')}")
    print(f"  - Session ID: {latest_state.values.get('session_id')}")
    
    # Get state history
    print("\nState History:")
    history = []
    for i, state_snapshot in enumerate(graph_compiled.get_state_history(config)):
        history.append(state_snapshot)
        if i < 3:  # Show last 3 states
            print(f"  State {i+1}:")
            print(f"    - Step: {state_snapshot.metadata.get('step', 'N/A') if state_snapshot.metadata else 'N/A'}")
            print(f"    - Next: {state_snapshot.next}")
    
    print(f"\nTotal checkpoints saved: {len(history)}")
    
except Exception as e:
    print(f"Error getting state history: {e}")


StateSnapshot Demonstration

Latest State:
  - Next node: ()
  - Messages count: 5
  - Summary: The conversation started with a casual greeting. The human asked why Mark (presumably Mark Zuckerber...
  - User ID: user_4ef024dc
  - Session ID: session_5b9a997e

State History:
  State 1:
    - Step: 9
    - Next: ()
  State 2:
    - Step: 8
    - Next: ('get_user_input',)
  State 3:
    - Step: 7
    - Next: ('summarize_and_store',)

Total checkpoints saved: 11


In [15]:
# ----------------------------
# Cell 15 — Demonstrate State Replay
# ----------------------------
print("\n" + "="*50)
print("State Replay Demonstration")
print("="*50)

try:
    # Get all checkpoints
    checkpoints = list(graph_compiled.get_state_history(config))
    
    if len(checkpoints) > 1:
        # Replay from a previous checkpoint (second last)
        checkpoint_to_replay = checkpoints[-2] if len(checkpoints) >= 2 else checkpoints[-1]
        
        print(f"\nReplaying from checkpoint at step: {checkpoint_to_replay.metadata.get('step', 'N/A') if checkpoint_to_replay.metadata else 'N/A'}")
        
        # Update config with checkpoint
        replay_config = {
            "configurable": {
                "thread_id": thread_id,
                "checkpoint_id": checkpoint_to_replay.config['configurable']['checkpoint_id']
            }
        }
        
        # Get state at that checkpoint
        replayed_state = graph_compiled.get_state(replay_config)
        print(f"Replayed state messages: {len(replayed_state.values.get('messages', []))}")
        
except Exception as e:
    print(f"Error in state replay: {e}")


State Replay Demonstration

Replaying from checkpoint at step: 0
Replayed state messages: 0


In [17]:
# ----------------------------
# Cell 16 — View Long-term Memory from SQLite
# ----------------------------
print("\n" + "="*50)
print("Long-term Memory from SQLite")
print("="*50)

# Load user memory
user_memory = load_user_memory(user_id)
print(f"\nUser ID: {user_id}")
print(f"Preferences: {user_memory['preferences']}")
print(f"Facts: {user_memory['facts']}")
print(f"Last Summary: {user_memory['summary'][:200]}..." if user_memory['summary'] else "No summary")

# Load recent conversations
recent_convs = load_conversation_from_db(user_id, session_id, limit=3)
print(f"\nRecent conversations: {len(recent_convs)}")
for i, conv in enumerate(recent_convs):
    print(f"\nConversation {i+1}:")
    print(f"  Messages: {len(conv['messages'])}")
    print(f"  Summary: {conv['summary'][:]}..." if conv['summary'] else "  No summary")


Long-term Memory from SQLite

User ID: user_4ef024dc
Preferences: {}
Facts: {}
Last Summary: The conversation started with a casual greeting. The human asked why Mark (presumably Mark Zuckerberg) diluted his partner's share to a very low percentage. The AI provided some background information...

Recent conversations: 1

Conversation 1:
  Messages: 7
  Summary: The conversation started with a casual greeting. The human asked why Mark (presumably Mark Zuckerberg) diluted his partner's share to a very low percentage. The AI provided some background information on the situation with Eduardo Saverin, but the human indicated that this was not the correct information. The conversation was cut off without further clarification on who the partner was or what specific situation the human was referring to....


In [20]:
# ----------------------------
# Cell 17 — Resume Previous Session
# ----------------------------
print("\n" + "="*50)
print("Resume Previous Session")
print("="*50)

# Ask for previous session ID
prev_session = input("\nEnter previous session ID to resume (or press Enter to skip): ").strip()

if prev_session:
    # Load previous session data
    prev_convs = load_conversation_from_db(user_id, prev_session, limit=1)
    
    if prev_convs:
        prev_conv = prev_convs[0]
        print(f"\nResuming session {prev_session}")
        print(f"Previous summary: {prev_conv['summary'][:200]}...")
        
        # Create new thread for resumed session
        new_thread_id = f"resume_{uuid.uuid4().hex[:8]}"
        resume_config = {"configurable": {"thread_id": new_thread_id}}
        
        # Start new session with previous context
        resume_state = State(
            messages=[],
            user_id=user_id,
            session_id=prev_session,
            summary=prev_conv['summary'],
            last_updated=datetime.now().isoformat(),
            metadata={"resumed": True, "previous_session": prev_session}
        )
        
        print("\nStarting resumed session...")
        # Uncomment to run resumed session
        # final_resume_state = graph_compiled.invoke(resume_state, resume_config)
    else:
        print(f"No session found with ID: {prev_session}")


Resume Previous Session

Resuming session session_5b9a997e
Previous summary: The conversation started with a casual greeting. The human asked why Mark (presumably Mark Zuckerberg) diluted his partner's share to a very low percentage. The AI provided some background information...

Starting resumed session...


In [21]:
# ----------------------------
# Cell 18 — Cleanup
# ----------------------------
# Close database connection
conn.close()
print("Database connection closed")

Database connection closed
